# TomTom GIS Distance and Travel-Time Collection for VNU Link Network

This Colab notebook processes the three input files:

1. `OD_matrix_links.xlsx` — OD link list.
2. `vnu_bus_stations.csv` — station coordinates.
3. `vnu_polygon_area.kmz` — VNU study-area polygon.

The notebook retrieves TomTom road-network route data for each link, using the following logic:

- First attempt: `travelMode = bus`.
- Second attempt for unresolved links: `travelMode = car`.
- Third attempt for unresolved links: omit `travelMode`, allowing TomTom to use its default mode.
- If no valid route is found after all attempts, the numeric outputs are left blank.

A route is accepted only when:

- The route geometry remains inside the VNU polygon.
- The route does not pass through any intermediate station, excluding the current link origin and destination.

In [1]:
# ============================================================
# 1. Install required packages
# ============================================================
!pip -q install pandas openpyxl requests shapely pyproj folium tqdm

In [2]:
# ============================================================
# 2. Import libraries
# ============================================================
import os
import time
import json
import math
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

from shapely.geometry import Point, LineString, Polygon, mapping
from shapely.ops import transform
from pyproj import Transformer

import folium

In [26]:
# ============================================================
# 3. User configuration
# ============================================================

# --- Input files ---
# Upload these files to /content/ in Colab before running the notebook.
OD_LINKS_FILE = "/content/OD_matrix_links.xlsx"
STATIONS_FILE = "/content/vnu_bus_stations.csv"
POLYGON_KMZ_FILE = "/content/vnu_polygon_area.kmz"

# --- TomTom API key ---
# Paste your TomTom Routing API key here.
TOMTOM_API_KEY = "YOUR_TOMTOM_API_KEY"

# --- Routing parameters ---
ROUTE_TYPE = "shortest"
COMPUTE_TRAVEL_TIME_FOR = "all"
TRAFFIC = "true"
MAX_ALTERNATIVES = 5

# Attempt order:
# 1) bus mode,
# 2) car fallback,
# 3) None = omit travelMode parameter and let TomTom use default mode.
TRAVEL_MODE_ATTEMPTS = ["bus", None]

# --- Spatial validation parameters ---
# EPSG:4326 = WGS84 lon/lat.
# EPSG:32648 = WGS84 / UTM zone 48N, suitable for Ho Chi Minh City distance calculations.
WGS84_CRS = "EPSG:4326"
METRIC_CRS = "EPSG:32648"

# Route may be very close to polygon boundary due to map-matching.
# A small tolerance avoids rejecting valid boundary-following routes.
POLYGON_TOLERANCE_M = 5.0

# If a station point is within this distance from the route line and lies between
# the route endpoints, it is treated as an intermediate station conflict.
STATION_INTERSECTION_BUFFER_M = 18.0

# Stations close to origin or destination are not treated as intermediate conflicts.
ENDPOINT_EXCLUSION_RADIUS_M = 45.0

# API safety controls
REQUEST_TIMEOUT_SECONDS = 30
API_SLEEP_SECONDS = 1.5

# Run switch.
# Keep False for the first run. After the single API test works, change to True.
RUN_FULL_COLLECTION = True

# --- Output folder ---
OUT_DIR = Path("/content/tomtom_vnu_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUT_DIR)

Output folder: /content/tomtom_vnu_outputs


In [ ]:
# ============================================================
# 4. Optional: upload files directly in Colab
# ============================================================
# Run this cell only if the files are not already in /content/.

try:
    from google.colab import files
    print("Colab upload tool is available.")
    print("Upload OD_matrix_links.xlsx, vnu_bus_stations.csv, and vnu_polygon_area.kmz if needed.")
    # uploaded = files.upload()
except Exception:
    print("Not running in Colab, or google.colab is not available.")

In [27]:
# ============================================================
# 5. Helper functions: input loading and standardization
# ============================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names while keeping readable field names."""
    df = df.copy()
    df.columns = [
        str(c).strip()
        .replace("\ufeff", "")
        .replace("\n", " ")
        .replace("\r", " ")
        for c in df.columns
    ]
    return df


def read_od_links(path: str) -> pd.DataFrame:
    """Read OD matrix links from Excel. Uses the first sheet by default."""
    xl = pd.ExcelFile(path)
    df = pd.read_excel(path, sheet_name=xl.sheet_names[0])
    df = normalize_columns(df)

    # Accept both naming styles if files are updated later.
    rename_map = {
        "Link Type": "link_type",
        "Link_name": "link_name",
        "from_station_id": "from_station_id",
        "to_station_id": "to_station_id",
    }
    df = df.rename(columns=rename_map)

    required = ["from_station_id", "to_station_id"]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Missing required OD link column: {col}")

    if "link_name" not in df.columns:
        df["link_name"] = df["from_station_id"].astype(str) + "_" + df["to_station_id"].astype(str)

    if "link_type" not in df.columns:
        df["link_type"] = ""

    df["from_station_id"] = df["from_station_id"].astype(int)
    df["to_station_id"] = df["to_station_id"].astype(int)

    return df


def read_stations(path: str) -> pd.DataFrame:
    """Read station coordinates."""
    df = pd.read_csv(path)
    df = normalize_columns(df)

    required = ["station_id", "longitude", "latitude"]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Missing required station column: {col}")

    if "station_name" not in df.columns:
        df["station_name"] = ""

    df["station_id"] = df["station_id"].astype(int)
    df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
    df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")

    if df[["longitude", "latitude"]].isna().any().any():
        bad = df[df[["longitude", "latitude"]].isna().any(axis=1)]
        raise ValueError(f"Station file contains invalid coordinates:\n{bad}")

    return df


def attach_station_coordinates(links_df: pd.DataFrame, stations_df: pd.DataFrame) -> pd.DataFrame:
    """Attach from/to longitude and latitude to each OD link."""
    stations_small = stations_df[["station_id", "longitude", "latitude", "station_name"]].copy()

    out = links_df.merge(
        stations_small.rename(columns={
            "station_id": "from_station_id",
            "longitude": "from_lon",
            "latitude": "from_lat",
            "station_name": "from_station_name",
        }),
        on="from_station_id",
        how="left",
    )

    out = out.merge(
        stations_small.rename(columns={
            "station_id": "to_station_id",
            "longitude": "to_lon",
            "latitude": "to_lat",
            "station_name": "to_station_name",
        }),
        on="to_station_id",
        how="left",
    )

    missing = out[out[["from_lon", "from_lat", "to_lon", "to_lat"]].isna().any(axis=1)]
    if len(missing) > 0:
        raise ValueError(
            "Some links cannot be matched with station coordinates. Check station IDs.\n"
            + str(missing[["from_station_id", "to_station_id"]].head(20))
        )

    return out


links_raw = read_od_links(OD_LINKS_FILE)
stations = read_stations(STATIONS_FILE)
links = attach_station_coordinates(links_raw, stations)

print("OD links:", links.shape)
display(links.head())

print("Stations:", stations.shape)
display(stations.head())

OD links: (307, 10)


,from_station_id,to_station_id,link_type,link_name,from_lon,from_lat,from_station_name,to_lon,to_lat,to_station_name
0,1,3,Forward,1_3,106.802862,10.877107,"International University, VNU-HCM",106.802491,10.876633,"International University, VNU-HCM - Creative S..."
1,1,2,U-turn,1_2,106.802862,10.877107,"International University, VNU-HCM",106.802752,10.878131,"International University, VNU-HCM - Le Quy Don..."
2,1,5,Forward,1_5,106.802862,10.877107,"International University, VNU-HCM",106.804914,10.875638,VNU-HCM Area A Dormitory - Creative Square Int...
3,1,25,Forward,1_25,106.802862,10.877107,"International University, VNU-HCM",106.802905,10.873613,VNU-HCM Area A Bus Terminal - Creative Square
4,2,1,U-turn,2_1,106.802752,10.878131,"International University, VNU-HCM - Le Quy Don...",106.802862,10.877107,"International University, VNU-HCM"


Stations: (92, 4)


,station_id,longitude,latitude,station_name
0,1,106.802862,10.877107,"International University, VNU-HCM"
1,2,106.802752,10.878131,"International University, VNU-HCM - Le Quy Don..."
2,3,106.802491,10.876633,"International University, VNU-HCM - Creative S..."
3,4,106.802486,10.876407,"International University, VNU-HCM - Opposite I..."
4,5,106.804914,10.875638,VNU-HCM Area A Dormitory - Creative Square Int...


In [28]:
# ============================================================
# 6. Helper functions: KMZ polygon loading
# ============================================================

def extract_kml_text_from_kmz(kmz_path: str) -> str:
    """Extract the first KML text file from a KMZ archive."""
    with zipfile.ZipFile(kmz_path, "r") as z:
        kml_files = [name for name in z.namelist() if name.lower().endswith(".kml")]
        if not kml_files:
            raise ValueError("No .kml file found inside the KMZ.")
        return z.read(kml_files[0]).decode("utf-8", errors="ignore")


def parse_first_polygon_from_kml(kml_text: str) -> Polygon:
    """Parse the first polygon coordinates from KML text into a Shapely polygon."""
    root = ET.fromstring(kml_text)

    # KML namespace handling
    ns = {"kml": "http://www.opengis.net/kml/2.2"}

    # Search standard KML polygon coordinates first.
    coord_nodes = root.findall(".//kml:Polygon//kml:outerBoundaryIs//kml:LinearRing//kml:coordinates", ns)

    # Fallback: any coordinates node.
    if not coord_nodes:
        coord_nodes = root.findall(".//kml:coordinates", ns)

    if not coord_nodes:
        raise ValueError("No polygon coordinates found in KML.")

    coord_text = coord_nodes[0].text.strip()

    coords = []
    for item in coord_text.split():
        parts = item.split(",")
        if len(parts) >= 2:
            lon = float(parts[0])
            lat = float(parts[1])
            coords.append((lon, lat))

    if len(coords) < 3:
        raise ValueError("Polygon has fewer than 3 coordinate points.")

    poly = Polygon(coords)

    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        raise ValueError("Parsed polygon is empty or invalid.")

    return poly


kml_text = extract_kml_text_from_kmz(POLYGON_KMZ_FILE)
study_area_wgs84 = parse_first_polygon_from_kml(kml_text)

print("Polygon loaded.")
print("Polygon bounds:", study_area_wgs84.bounds)


Polygon loaded.
Polygon bounds: (106.764206, 10.8603504, 106.8154156, 10.8945102)


In [29]:
# ============================================================
# 7. Coordinate transformation and spatial objects
# ============================================================

to_metric_transformer = Transformer.from_crs(WGS84_CRS, METRIC_CRS, always_xy=True)
to_wgs84_transformer = Transformer.from_crs(METRIC_CRS, WGS84_CRS, always_xy=True)

def to_metric_geom(geom):
    return transform(to_metric_transformer.transform, geom)

def to_wgs84_geom(geom):
    return transform(to_wgs84_transformer.transform, geom)

study_area_utm = to_metric_geom(study_area_wgs84)
study_area_utm_buffered = study_area_utm.buffer(POLYGON_TOLERANCE_M)

station_points_wgs84 = {
    int(row.station_id): Point(float(row.longitude), float(row.latitude))
    for row in stations.itertuples(index=False)
}

station_points_utm = {
    sid: to_metric_geom(pt)
    for sid, pt in station_points_wgs84.items()
}

print("Metric polygon area, km²:", study_area_utm.area / 1_000_000)
print("Number of station points:", len(station_points_utm))

Metric polygon area, km²: 12.876137251344437
Number of station points: 92


In [33]:
# ============================================================
# 8. TomTom API functions
# ============================================================

def travel_mode_label(mode: Optional[str]) -> str:
    if mode is None:
        return "tomtom_default"
    return str(mode)


def build_tomtom_params(api_key: str, travel_mode: Optional[str]) -> Dict[str, Any]:
    """Build TomTom request parameters.

    If travel_mode is None, the travelMode parameter is omitted.
    """
    params = {
        "key": api_key,
        "routeType": ROUTE_TYPE,
        "traffic": TRAFFIC,
        "computeTravelTimeFor": COMPUTE_TRAVEL_TIME_FOR,
        "routeRepresentation": "polyline",
        "maxAlternatives": MAX_ALTERNATIVES,
    }

    if travel_mode is not None:
        params["travelMode"] = travel_mode

    return params


def calculate_tomtom_route(
    from_lat: float,
    from_lon: float,
    to_lat: float,
    to_lon: float,
    travel_mode: Optional[str],
    api_key: str,
) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """Call TomTom Calculate Route API for one OD pair.

    Returns:
        response_json, error_message
    """
    if not api_key or api_key == "PASTE_YOUR_TOMTOM_API_KEY_HERE":
        return None, "TOMTOM_API_KEY is missing. Paste your API key in the configuration cell."

    url = (
        "https://api.tomtom.com/routing/1/calculateRoute/"
        f"{from_lat},{from_lon}:{to_lat},{to_lon}/json"
    )

    params = build_tomtom_params(api_key, travel_mode)

    # Retry logic for HTTP 429 Too Many Requests.
    response = None
    last_429_message = ""

    for retry in range(5):
        try:
            response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        except requests.RequestException as e:
            return None, f"REQUEST_EXCEPTION: {e}"

        if response.status_code == 429:
            wait_time = 10 * (retry + 1)
            try:
                last_429_message = json.dumps(response.json())[:500]
            except Exception:
                last_429_message = response.text[:500]

            print(f"HTTP 429 Too Many Requests. Waiting {wait_time} seconds...")
            time.sleep(wait_time)
            continue

        break

    if response is None:
        return None, "NO_RESPONSE"

    if response.status_code == 429:
        return None, f"HTTP_429_AFTER_5_RETRIES: {last_429_message}"

    if response.status_code != 200:
        try:
            err_json = response.json()
            err_msg = json.dumps(err_json)[:500]
        except Exception:
            err_msg = response.text[:500]
        return None, f"HTTP_{response.status_code}: {err_msg}"

    try:
        return response.json(), None
    except Exception as e:
        return None, f"JSON_PARSE_ERROR: {e}"


def route_json_to_linestring(route: Dict[str, Any]) -> Optional[LineString]:
    """Convert TomTom route geometry to Shapely LineString in WGS84 lon/lat."""
    points_lonlat = []

    for leg in route.get("legs", []):
        for p in leg.get("points", []):
            if "longitude" in p and "latitude" in p:
                lonlat = (float(p["longitude"]), float(p["latitude"]))
                if not points_lonlat or lonlat != points_lonlat[-1]:
                    points_lonlat.append(lonlat)

    if len(points_lonlat) < 2:
        return None

    return LineString(points_lonlat)


def get_average_time_seconds(summary: Dict[str, Any]) -> Tuple[Optional[float], str]:
    """Get historical average travel time if available; otherwise use best-estimate travel time."""
    if "historicTrafficTravelTimeInSeconds" in summary:
        return float(summary["historicTrafficTravelTimeInSeconds"]), "historicTrafficTravelTimeInSeconds"

    if "noTrafficTravelTimeInSeconds" in summary:
        # If historical traffic is unavailable, no-traffic time is at least a stable average-like reference.
        return float(summary["noTrafficTravelTimeInSeconds"]), "noTrafficTravelTimeInSeconds"

    if "travelTimeInSeconds" in summary:
        return float(summary["travelTimeInSeconds"]), "travelTimeInSeconds"

    return None, "missing_time_field"

In [34]:
# ============================================================
# 9. Route validation functions
# ============================================================

def check_route_inside_polygon(route_line_wgs84: LineString) -> Tuple[bool, str]:
    """Check whether route is inside the VNU polygon with tolerance."""
    route_line_utm = to_metric_geom(route_line_wgs84)

    if study_area_utm_buffered.covers(route_line_utm):
        return True, "inside_polygon"

    outside_length_m = route_line_utm.difference(study_area_utm_buffered).length
    return False, f"outside_polygon_length_m={outside_length_m:.2f}"


def find_intermediate_station_conflicts(
    route_line_wgs84: LineString,
    from_station_id: int,
    to_station_id: int,
) -> List[Dict[str, Any]]:
    """Find stations located in the middle of the route.

    The origin and destination stations are excluded.
    Stations near the route endpoints are also ignored because they are not middle-route conflicts.
    """
    route_line_utm = to_metric_geom(route_line_wgs84)
    route_len = route_line_utm.length

    conflicts = []
    exclude_ids = {int(from_station_id), int(to_station_id)}

    for sid, pt_utm in station_points_utm.items():
        if sid in exclude_ids:
            continue

        dist_m = route_line_utm.distance(pt_utm)

        if dist_m <= STATION_INTERSECTION_BUFFER_M:
            along_m = route_line_utm.project(pt_utm)

            is_middle = (
                along_m > ENDPOINT_EXCLUSION_RADIUS_M
                and along_m < route_len - ENDPOINT_EXCLUSION_RADIUS_M
            )

            if is_middle:
                station_name = stations.loc[stations["station_id"] == sid, "station_name"]
                station_name = station_name.iloc[0] if len(station_name) else ""

                conflicts.append({
                    "station_id": sid,
                    "station_name": station_name,
                    "distance_to_route_m": round(float(dist_m), 3),
                    "position_along_route_m": round(float(along_m), 3),
                })

    return conflicts


def validate_route(
    route_line_wgs84: Optional[LineString],
    from_station_id: int,
    to_station_id: int,
) -> Tuple[bool, str, List[Dict[str, Any]]]:
    """Apply polygon and intermediate-station validation."""
    if route_line_wgs84 is None:
        return False, "missing_route_geometry", []

    inside, polygon_msg = check_route_inside_polygon(route_line_wgs84)
    if not inside:
        return False, polygon_msg, []

    conflicts = find_intermediate_station_conflicts(
        route_line_wgs84,
        from_station_id=from_station_id,
        to_station_id=to_station_id,
    )

    if conflicts:
        conflict_ids = [str(c["station_id"]) for c in conflicts[:10]]
        msg = "intermediate_station_conflict=" + ",".join(conflict_ids)
        if len(conflicts) > 10:
            msg += f"_and_{len(conflicts)-10}_more"
        return False, msg, conflicts

    return True, "valid_route", []

In [35]:
# ============================================================
# 10. Select the first valid TomTom route among alternatives
# ============================================================

def select_valid_route_from_response(
    response_json: Dict[str, Any],
    from_station_id: int,
    to_station_id: int,
) -> Tuple[Optional[Dict[str, Any]], Optional[LineString], str, List[Dict[str, Any]]]:
    """Return the first route alternative satisfying all GIS validation rules."""
    routes = response_json.get("routes", [])

    if not routes:
        return None, None, "no_routes_returned", []

    validation_messages = []

    for idx, route in enumerate(routes):
        route_line = route_json_to_linestring(route)

        valid, msg, conflicts = validate_route(
            route_line,
            from_station_id=from_station_id,
            to_station_id=to_station_id,
        )

        if valid:
            return route, route_line, f"selected_candidate_index={idx}", []

        validation_messages.append(f"candidate_{idx}:{msg}")

    return None, None, " | ".join(validation_messages), []


def build_result_row_from_route(
    link_row: pd.Series,
    selected_route: Dict[str, Any],
    route_line_wgs84: LineString,
    travel_mode_used: Optional[str],
    validation_detail: str,
) -> Dict[str, Any]:
    """Build one accepted output record."""
    summary = selected_route.get("summary", {})

    distance_m = summary.get("lengthInMeters", np.nan)
    distance_km = float(distance_m) / 1000 if pd.notna(distance_m) else np.nan

    avg_time_s, avg_time_source = get_average_time_seconds(summary)
    avg_time_min = avg_time_s / 60 if avg_time_s is not None else np.nan

    avg_speed_kmh = np.nan
    if avg_time_s is not None and avg_time_s > 0 and pd.notna(distance_km):
        avg_speed_kmh = distance_km / (avg_time_s / 3600)

    return {
        "from_station_id": int(link_row["from_station_id"]),
        "to_station_id": int(link_row["to_station_id"]),
        "link_name": link_row.get("link_name", ""),
        "link_type": link_row.get("link_type", ""),
        "from_lat": float(link_row["from_lat"]),
        "from_lon": float(link_row["from_lon"]),
        "to_lat": float(link_row["to_lat"]),
        "to_lon": float(link_row["to_lon"]),
        "from_station_name": link_row.get("from_station_name", ""),
        "to_station_name": link_row.get("to_station_name", ""),

        "validation_status": "VALID",
        "validation_detail": validation_detail,
        "travel_mode_used": travel_mode_label(travel_mode_used),

        "gis_route_distance_m": distance_m,
        "gis_route_distance_km": distance_km,

        "average_travel_time_s": avg_time_s,
        "average_travel_time_min": avg_time_min,
        "average_travel_time_source": avg_time_source,

        "average_speed_kmh": avg_speed_kmh,

        "tomtom_travel_time_s": summary.get("travelTimeInSeconds", np.nan),
        "tomtom_no_traffic_time_s": summary.get("noTrafficTravelTimeInSeconds", np.nan),
        "tomtom_historic_traffic_time_s": summary.get("historicTrafficTravelTimeInSeconds", np.nan),
        "tomtom_live_traffic_time_s": summary.get("liveTrafficIncidentsTravelTimeInSeconds", np.nan),
        "tomtom_traffic_delay_s": summary.get("trafficDelayInSeconds", np.nan),

        "route_wkt": route_line_wgs84.wkt,
        "route_geojson": json.dumps(mapping(route_line_wgs84)),
        "api_error": "",
    }


def build_blank_result_row(
    link_row: pd.Series,
    validation_detail: str,
    api_error: str = "",
) -> Dict[str, Any]:
    """Build one unresolved output record with blank numerical fields."""
    return {
        "from_station_id": int(link_row["from_station_id"]),
        "to_station_id": int(link_row["to_station_id"]),
        "link_name": link_row.get("link_name", ""),
        "link_type": link_row.get("link_type", ""),
        "from_lat": float(link_row["from_lat"]),
        "from_lon": float(link_row["from_lon"]),
        "to_lat": float(link_row["to_lat"]),
        "to_lon": float(link_row["to_lon"]),
        "from_station_name": link_row.get("from_station_name", ""),
        "to_station_name": link_row.get("to_station_name", ""),

        "validation_status": "NO_VALID_ROUTE_AFTER_ALL_MODES",
        "validation_detail": validation_detail,
        "travel_mode_used": "",

        "gis_route_distance_m": np.nan,
        "gis_route_distance_km": np.nan,

        "average_travel_time_s": np.nan,
        "average_travel_time_min": np.nan,
        "average_travel_time_source": "",

        "average_speed_kmh": np.nan,

        "tomtom_travel_time_s": np.nan,
        "tomtom_no_traffic_time_s": np.nan,
        "tomtom_historic_traffic_time_s": np.nan,
        "tomtom_live_traffic_time_s": np.nan,
        "tomtom_traffic_delay_s": np.nan,

        "route_wkt": "",
        "route_geojson": "",
        "api_error": api_error,
    }

In [36]:
# ============================================================
# 11. Single API test section
# ============================================================

def test_one_tomtom_request(test_mode: Optional[str] = "bus") -> None:
    """Test one TomTom request using the first OD link."""
    row = links.iloc[0]

    print("Testing one TomTom request")
    print("Link:", row["from_station_id"], "->", row["to_station_id"])
    print("Mode:", travel_mode_label(test_mode))

    response_json, error = calculate_tomtom_route(
        from_lat=float(row["from_lat"]),
        from_lon=float(row["from_lon"]),
        to_lat=float(row["to_lat"]),
        to_lon=float(row["to_lon"]),
        travel_mode=test_mode,
        api_key=TOMTOM_API_KEY,
    )

    if error:
        print("API test failed:")
        print(error)
        return

    routes = response_json.get("routes", [])
    print("API test successful.")
    print("Number of routes returned:", len(routes))

    if routes:
        summary = routes[0].get("summary", {})
        print("First route summary:")
        print(json.dumps(summary, indent=2)[:1500])

        selected_route, route_line, detail, conflicts = select_valid_route_from_response(
            response_json,
            from_station_id=int(row["from_station_id"]),
            to_station_id=int(row["to_station_id"]),
        )

        print("GIS validation result:", "VALID" if selected_route is not None else "INVALID")
        print("Validation detail:", detail)


# Run the single request test.
# Change "bus" to "car" or None if you want to test another mode.
test_one_tomtom_request(test_mode="bus")

Testing one TomTom request
Link: 1 -> 3
Mode: bus
API test successful.
Number of routes returned: 1
First route summary:
{
  "lengthInMeters": 142,
  "travelTimeInSeconds": 23,
  "trafficDelayInSeconds": 0,
  "trafficLengthInMeters": 0,
  "departureTime": "2026-04-30T17:31:43+07:00",
  "arrivalTime": "2026-04-30T17:32:06+07:00",
  "noTrafficTravelTimeInSeconds": 21,
  "historicTrafficTravelTimeInSeconds": 23,
  "liveTrafficIncidentsTravelTimeInSeconds": 23
}
GIS validation result: VALID
Validation detail: selected_candidate_index=0


In [37]:
# ============================================================
# 12. Batch processing function
# ============================================================

def process_one_link(link_row: pd.Series) -> Dict[str, Any]:
    """Process one OD link using bus first, then car/default fallback."""
    all_attempt_messages = []
    api_errors = []

    for travel_mode in TRAVEL_MODE_ATTEMPTS:
        response_json, error = calculate_tomtom_route(
            from_lat=float(link_row["from_lat"]),
            from_lon=float(link_row["from_lon"]),
            to_lat=float(link_row["to_lat"]),
            to_lon=float(link_row["to_lon"]),
            travel_mode=travel_mode,
            api_key=TOMTOM_API_KEY,
        )

        mode_label = travel_mode_label(travel_mode)

        if error:
            msg = f"{mode_label}: {error}"
            all_attempt_messages.append(msg)
            api_errors.append(msg)
            time.sleep(API_SLEEP_SECONDS)
            continue

        selected_route, route_line, detail, conflicts = select_valid_route_from_response(
            response_json=response_json,
            from_station_id=int(link_row["from_station_id"]),
            to_station_id=int(link_row["to_station_id"]),
        )

        if selected_route is not None:
            return build_result_row_from_route(
                link_row=link_row,
                selected_route=selected_route,
                route_line_wgs84=route_line,
                travel_mode_used=travel_mode,
                validation_detail=detail,
            )

        all_attempt_messages.append(f"{mode_label}: {detail}")
        time.sleep(API_SLEEP_SECONDS)

    return build_blank_result_row(
        link_row=link_row,
        validation_detail=" || ".join(all_attempt_messages),
        api_error=" || ".join(api_errors),
    )


def run_batch_collection(links_df: pd.DataFrame) -> pd.DataFrame:
    """Run TomTom routing collection for all OD links."""
    results = []

    for _, row in tqdm(links_df.iterrows(), total=len(links_df), desc="Collecting TomTom routes"):
        result = process_one_link(row)
        results.append(result)

    return pd.DataFrame(results)

In [38]:
# ============================================================
# 13. Run full collection
# ============================================================

if RUN_FULL_COLLECTION:
    results_df = run_batch_collection(links)
    print("Collection finished.")
    display(results_df.head())
else:
    print("RUN_FULL_COLLECTION is False.")
    print("After the single API test works, set RUN_FULL_COLLECTION = True in the configuration cell and rerun from that cell.")
    results_df = pd.DataFrame()

HTTP 429 Too Many Requests. Waiting 10 seconds...
HTTP 429 Too Many Requests. Waiting 10 seconds...
HTTP 429 Too Many Requests. Waiting 10 seconds...
HTTP 429 Too Many Requests. Waiting 10 seconds...
HTTP 429 Too Many Requests. Waiting 10 seconds...
HTTP 429 Too Many Requests. Waiting 10 seconds...
HTTP 429 Too Many Requests. Waiting 10 seconds...
HTTP 429 Too Many Requests. Waiting 10 seconds...
Collection finished.


,from_station_id,to_station_id,link_name,link_type,from_lat,from_lon,to_lat,to_lon,from_station_name,to_station_name,...,average_travel_time_source,average_speed_kmh,tomtom_travel_time_s,tomtom_no_traffic_time_s,tomtom_historic_traffic_time_s,tomtom_live_traffic_time_s,tomtom_traffic_delay_s,route_wkt,route_geojson,api_error
0,1,3,1_3,Forward,10.877107,106.802862,10.876633,106.802491,"International University, VNU-HCM","International University, VNU-HCM - Creative S...",...,historicTrafficTravelTimeInSeconds,22.226087,23.0,21.0,23.0,23.0,0.0,"LINESTRING (106.80298 10.87715, 106.80318 10.8...","{""type"": ""LineString"", ""coordinates"": [[106.80...",
1,1,2,1_2,U-turn,10.877107,106.802862,10.878131,106.802752,"International University, VNU-HCM","International University, VNU-HCM - Le Quy Don...",...,historicTrafficTravelTimeInSeconds,24.975000,16.0,14.0,16.0,16.0,0.0,"LINESTRING (106.80298 10.87715, 106.80286 10.8...","{""type"": ""LineString"", ""coordinates"": [[106.80...",
2,1,5,1_5,Forward,10.877107,106.802862,10.875638,106.804914,"International University, VNU-HCM",VNU-HCM Area A Dormitory - Creative Square Int...,...,historicTrafficTravelTimeInSeconds,23.591489,47.0,40.0,47.0,47.0,0.0,"LINESTRING (106.80298 10.87715, 106.80318 10.8...","{""type"": ""LineString"", ""coordinates"": [[106.80...",
3,1,25,1_25,Forward,10.877107,106.802862,10.873613,106.802905,"International University, VNU-HCM",VNU-HCM Area A Bus Terminal - Creative Square,...,,NaN,NaN,NaN,NaN,NaN,NaN,,,
4,2,1,2_1,U-turn,10.878131,106.802752,10.877107,106.802862,"International University, VNU-HCM - Le Quy Don...","International University, VNU-HCM",...,historicTrafficTravelTimeInSeconds,28.542857,14.0,12.0,14.0,14.0,0.0,"LINESTRING (106.80265 10.8781, 106.80286 10.87...","{""type"": ""LineString"", ""coordinates"": [[106.80...",


In [39]:
# ============================================================
# 14. Export results: CSV, Excel, GeoJSON, and HTML map
# ============================================================

def export_geojson(results_df: pd.DataFrame, path: Path) -> None:
    """Export valid route geometries as GeoJSON FeatureCollection."""
    features = []

    valid_df = results_df[
        (results_df["validation_status"] == "VALID")
        & (results_df["route_geojson"].astype(str).str.len() > 0)
    ].copy()

    for _, row in valid_df.iterrows():
        geom = json.loads(row["route_geojson"])
        props = row.drop(labels=["route_geojson"]).to_dict()

        # Keep GeoJSON properties compact.
        props.pop("route_wkt", None)

        features.append({
            "type": "Feature",
            "geometry": geom,
            "properties": props,
        })

    fc = {
        "type": "FeatureCollection",
        "features": features,
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False, indent=2)


def export_html_map(results_df: pd.DataFrame, stations_df: pd.DataFrame, polygon_wgs84: Polygon, path: Path) -> None:
    """Export an HTML map for route checking."""
    centroid = polygon_wgs84.centroid
    m = folium.Map(location=[centroid.y, centroid.x], zoom_start=15, tiles="OpenStreetMap")

    # Polygon layer
    poly_coords_latlon = [(lat, lon) for lon, lat in list(polygon_wgs84.exterior.coords)]
    folium.Polygon(
        locations=poly_coords_latlon,
        popup="VNU Polygon",
        fill=False,
        weight=2,
    ).add_to(m)

    # Station layer
    for _, st in stations_df.iterrows():
        folium.CircleMarker(
            location=[st["latitude"], st["longitude"]],
            radius=3,
            popup=f'{st["station_id"]}: {st.get("station_name", "")}',
            fill=True,
        ).add_to(m)

    # Route layer
    if len(results_df) > 0:
        valid_df = results_df[
            (results_df["validation_status"] == "VALID")
            & (results_df["route_geojson"].astype(str).str.len() > 0)
        ].copy()

        for _, row in valid_df.iterrows():
            geom = json.loads(row["route_geojson"])
            coords_latlon = [(lat, lon) for lon, lat in geom["coordinates"]]

            popup = (
                f'Link: {row["link_name"]}<br>'
                f'Mode: {row["travel_mode_used"]}<br>'
                f'Distance km: {row["gis_route_distance_km"]:.3f}<br>'
                f'Avg time min: {row["average_travel_time_min"]:.3f}<br>'
                f'Avg speed km/h: {row["average_speed_kmh"]:.2f}'
            )

            folium.PolyLine(
                locations=coords_latlon,
                weight=3,
                popup=popup,
            ).add_to(m)

    m.save(str(path))


if len(results_df) > 0:
    csv_path = OUT_DIR / "tomtom_vnu_link_results.csv"
    xlsx_path = OUT_DIR / "tomtom_vnu_link_results.xlsx"
    geojson_path = OUT_DIR / "tomtom_vnu_link_results.geojson"
    html_path = OUT_DIR / "tomtom_vnu_link_map.html"

    results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    results_df.to_excel(xlsx_path, index=False)
    export_geojson(results_df, geojson_path)
    export_html_map(results_df, stations, study_area_wgs84, html_path)

    print("Saved outputs:")
    print(csv_path)
    print(xlsx_path)
    print(geojson_path)
    print(html_path)
else:
    print("No results to export yet. Run full collection first.")

Saved outputs:
/content/tomtom_vnu_outputs/tomtom_vnu_link_results.csv
/content/tomtom_vnu_outputs/tomtom_vnu_link_results.xlsx
/content/tomtom_vnu_outputs/tomtom_vnu_link_results.geojson
/content/tomtom_vnu_outputs/tomtom_vnu_link_map.html


In [40]:
# ============================================================
# 15. Summary tables
# ============================================================

if len(results_df) > 0:
    print("Validation status summary:")
    display(results_df["validation_status"].value_counts(dropna=False).to_frame("count"))

    print("Travel mode used summary:")
    display(results_df["travel_mode_used"].replace("", "blank").value_counts(dropna=False).to_frame("count"))

    print("Unresolved links:")
    unresolved = results_df[results_df["validation_status"] != "VALID"].copy()
    display(unresolved[[
        "from_station_id",
        "to_station_id",
        "link_name",
        "link_type",
        "validation_detail",
        "api_error",
    ]].head(50))
else:
    print("No results available yet.")

Validation status summary:


,count
validation_status,
VALID,205
NO_VALID_ROUTE_AFTER_ALL_MODES,102


Travel mode used summary:


,count
travel_mode_used,
bus,193
blank,102
tomtom_default,12


Unresolved links:


,from_station_id,to_station_id,link_name,link_type,validation_detail,api_error
3,1,25,1_25,Forward,bus: candidate_0:intermediate_station_conflict...,
5,2,24,2_24,Forward,bus: candidate_0:intermediate_station_conflict...,
12,2,35,2_35,Forward,bus: candidate_0:outside_polygon_length_m=213....,
18,4,2,4_2,Forward,bus: candidate_0:intermediate_station_conflict...,
20,4,25,4_25,Forward,bus: candidate_0:intermediate_station_conflict...,
26,6,2,6_2,Forward,bus: candidate_0:intermediate_station_conflict...,
28,6,25,6_25,Forward,bus: candidate_0:intermediate_station_conflict...,
30,7,33,7_33,Forward,bus: candidate_0:intermediate_station_conflict...,
33,8,6,8_6,Forward,bus: candidate_0:intermediate_station_conflict...,
34,8,25,8_25,Forward,bus: candidate_0:intermediate_station_conflict...,


In [ ]:
# ============================================================
# 16. Download outputs from Colab
# ============================================================
# Run this cell after full collection and export.

try:
    from google.colab import files

    for p in [
        OUT_DIR / "tomtom_vnu_link_results.csv",
        OUT_DIR / "tomtom_vnu_link_results.xlsx",
        OUT_DIR / "tomtom_vnu_link_results.geojson",
        OUT_DIR / "tomtom_vnu_link_map.html",
    ]:
        if p.exists():
            print("Downloading:", p.name)
            files.download(str(p))
except Exception:
    print("Download helper is available only in Colab.")